In [8]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(sys.path[0])

d:\internship project\multidoc-rag


In [2]:
!pip install nltk

  Using cached defusedxml-0.7.1-py2.py3-none-any.whl.metadata (32 kB)
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 11.7 MB/s  0:00:00
Using cached defusedxml-0.7.1-py2.py3-none-any.whl (25 kB)

   ---------------------------------------- 0/2 [defusedxml]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   -------------------- ------------------- 1/2 [nltk]
   ---


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from nltk.tokenize import sent_tokenize
from typing import List

In [4]:
import nltk
nltk.download("punkt")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Harshita\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


True

In [5]:
from nltk.tokenize import sent_tokenize


def split_chunks_into_sentences(retrieval_results):
    """
    Convert retrieved chunks into individual candidate sentences.
    """

    candidate_sentences = []

    for result in retrieval_results:

        chunk = result.document.page_content
        metadata = result.document.metadata

        for sentence in sent_tokenize(chunk):

            sentence = sentence.strip()

            if len(sentence) < 15:
                continue

            candidate_sentences.append(
                {
                    "sentence": sentence,
                    "metadata": metadata,
                    "retrieval_score": result.retrieval_score,
                    "chunk_rank": result.rank,
                }
            )

    return candidate_sentences

In [10]:
from pathlib import Path

EXPERIMENT_NAME = "hybrid_rrf_crossencoder_recursive700"

CHUNK_SIZE = 700
CHUNK_OVERLAP = 100

EXPERIMENT_DIR = Path(f"experiments/{EXPERIMENT_NAME}")

VECTOR_DB = EXPERIMENT_DIR / "chroma_db"
RETRIEVAL_RESULTS = EXPERIMENT_DIR / "retrieval_results.csv"
METRICS = EXPERIMENT_DIR / "metrics.csv"

EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

print(EXPERIMENT_DIR.resolve())

D:\internship project\multidoc-rag\src\experiments\hybrid_rrf_crossencoder_recursive700


In [12]:
from src.evaluation.benchmark import BenchmarkDataset
from src.evaluation.evaluator import RetrievalEvaluator

benchmark = BenchmarkDataset(
    "../data/benchmark/benchmark_queries.csv"
)

evaluator = RetrievalEvaluator(benchmark)

In [13]:
from src.ingestion import load_documents
from src.chunking import split_documents
from src.retrievers.hybrid_rrf import HybridRRFRetriever   

documents = load_documents("../data/corpus")

chunks = split_documents(
    documents,
    chunk_size=700,
    chunk_overlap=100,
    chunking_method="recursive",
)

retriever = HybridRRFRetriever(
    chunks=chunks,
    persist_directory=VECTOR_DB,
    k=5,
)

results = evaluator.evaluate(retriever)

Found 2 PDF files.
Loaded 36 PDF pages.

Found 1027 Markdown files.
Loaded 1027 Markdown documents.

Total documents loaded: 1063


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading Cross Encoder: BAAI/bge-reranker-base


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [27]:
import json

with open("ragas_dataset.json", "r", encoding="utf-8") as f:
    evaluation_rows = json.load(f)

question = evaluation_rows[0]["question"]

FileNotFoundError: [Errno 2] No such file or directory: 'ragas_dataset.json'

In [19]:
response = retriever.search_with_scores(question)

In [21]:
response = retriever.search_with_scores(question)

In [22]:
question = reference_df.iloc[0]["query"]

response = retriever.search_with_scores(question)

candidate_sentences = split_chunks_into_sentences(response.results)

sentence_text = [
    s["sentence"]
    for s in candidate_sentences
]

scored = retriever.reranker.score_sentences(
    question,
    sentence_text,
)

scored = sorted(
    scored,
    key=lambda x: x[1],
    reverse=True,
)

for sentence, score in scored[:10]:
    print(f"{score:.3f}  {sentence}")

NameError: name 'reference_df' is not defined

In [23]:
%whos

Variable                      Type                    Data/Info
---------------------------------------------------------------
BenchmarkDataset              type                    <class 'src.evaluation.be<...>chmark.BenchmarkDataset'>
CHUNK_OVERLAP                 int                     100
CHUNK_SIZE                    int                     700
EXPERIMENT_DIR                WindowsPath             experiments\hybrid_rrf_crossencoder_recursive700
EXPERIMENT_NAME               str                     hybrid_rrf_crossencoder_recursive700
HybridRRFRetriever            type                    <class 'src.retrievers.hy<...>_rrf.HybridRRFRetriever'>
List                          _SpecialGenericAlias    typing.List
METRICS                       WindowsPath             experiments\hybrid_rrf_cr<...>_recursive700\metrics.csv
PROJECT_ROOT                  WindowsPath             d:\internship project\multidoc-rag
Path                          type                    <class 'pathlib.Path'>


In [24]:
print(type(retriever))


<class 'src.retrievers.hybrid_rrf.HybridRRFRetriever'>
